# Complete Chapter 9 Betting Workflow on Real Data

This extra notebook syncs the full chapter-9 workflow with the real Football-Data CSV. Rather than building on synthetic predictions, it uses the same calibrated probabilities and chronological holdout setup as the book and the main notebook.

In [ ]:
import importlib.util
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
ROOT = next(path for path in [cwd, *cwd.parents] if (path / 'Companion-Code').exists())
MODULE_PATH = ROOT / 'Companion-Code' / 'extras' / 'chapter-9' / 'real_data_betting_workflow.py'
spec = importlib.util.spec_from_file_location('chapter9_real', MODULE_PATH)
chapter9_real = importlib.util.module_from_spec(spec)
spec.loader.exec_module(chapter9_real)

matches = chapter9_real.load_matches()
train_df, valid_df = chapter9_real.train_validation_split(matches)
base_model, calibrated_model = chapter9_real.build_calibrated_model(train_df)
summary = chapter9_real.evaluate_model(base_model, calibrated_model, valid_df)
summary_table = pd.Series({
    'Matches': len(matches),
    'Train Matches': len(train_df),
    'Validation Matches': len(valid_df),
    'Flat Value Bets': summary['flat_value_bets'],
    'Flat Value ROI': summary['flat_value_roi'],
    'Kelly ROI': summary['kelly_roi'],
})
summary_table


## Compact System Report

In [ ]:
report = pd.DataFrame({
    'Metric': [
        'Base Accuracy', 'Base Log Loss', 'Calibrated Accuracy', 'Calibrated Log Loss',
        'Flat Value Bets', 'Flat Value Win Rate', 'Flat Value Profit', 'Flat Value ROI',
        'Kelly Bets', 'Kelly Final Bankroll', 'Kelly Profit', 'Kelly ROI',
    ],
    'Value': [
        summary['base_accuracy'], summary['base_log_loss'], summary['calibrated_accuracy'], summary['calibrated_log_loss'],
        summary['flat_value_bets'], summary['flat_value_win_rate'], summary['flat_value_profit'], summary['flat_value_roi'],
        summary['kelly_bets'], summary['kelly_final_bankroll'], summary['kelly_profit'], summary['kelly_roi'],
    ],
})
report


## Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(matches['Kickoff'], chapter9_real.baseline_paths(matches)['home']['roi_path'], color='#0B5D7A', linewidth=2)
axes[0, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[0, 0].set_title('Always Home ROI')
axes[0, 0].grid(alpha=0.25)

axes[0, 1].plot(matches['Kickoff'], chapter9_real.baseline_paths(matches)['favorite']['roi_path'], color='#B63A1B', linewidth=2)
axes[0, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[0, 1].set_title('Always Favorite ROI')
axes[0, 1].grid(alpha=0.25)

axes[1, 0].plot(valid_df['Kickoff'], summary['flat_roi_path'], linewidth=2, color='#0B5D7A', label='Flat Value Bets')
axes[1, 0].plot(valid_df['Kickoff'], summary['kelly_roi_path'], linewidth=2, color='#B63A1B', label='Full Kelly')
axes[1, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[1, 0].set_title('Validation ROI Paths')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

metrics = pd.Series({
    'Flat ROI': summary['flat_value_roi'],
    'Kelly ROI': summary['kelly_roi'],
    'Flat Profit': summary['flat_value_profit'],
    'Kelly Profit': summary['kelly_profit'],
})
axes[1, 1].bar(metrics.index, metrics.values, color=['#0B5D7A', '#B63A1B', '#0B5D7A', '#B63A1B'])
axes[1, 1].axhline(0, color='gray', linewidth=1)
axes[1, 1].set_title('Headline Betting Outcomes')
axes[1, 1].tick_params(axis='x', rotation=30)
axes[1, 1].grid(alpha=0.25, axis='y')

plt.tight_layout()
plt.show()


## One-Command Regeneration

If you want to regenerate the saved figures and summary files outside the notebook, run `Companion-Code/extras/chapter-9/real_data_betting_workflow.py` directly.